# Evaluating A/B testing results with Python

In [ ]:
# Loading the data
import pandas as pd
import janitor
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

df = pd.read_csv('../../data/raw/WA_Fn-UseC_-Marketing-Campaign-Eff-UseC_-FastF.csv').clean_names(case_type='snake')    

In [ ]:
df.shape

In [ ]:
df.head(15)

# Data Analysis

#### - Total Sales

In [ ]:
df['sales_in_thousands'].describe()

In [ ]:
import plotly.express as px

sales_by_promo = (
    df.groupby('promotion', as_index=False)['sales_in_thousands']
    .sum()
)

fig = px.pie(
    sales_by_promo,
    names='promotion',
    values='sales_in_thousands',
    title='sales distribution across different promotions',
    template='ggplot2'
)

fig.update_traces(texttemplate='%{percent:.0%}', textposition='inside')
fig.show()

#### - Market Size

In [ ]:
df.groupby('market_size').count()['market_id']

In [ ]:
market_counts = (
    df.groupby(['promotion', 'market_size'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig = px.bar(
    market_counts,
    x='promotion',
    y='count',
    color='market_size',
    barmode='group',
    title='breakdowns of market sizes across different promotions',
    labels={'promotion': 'promotion', 'count': 'count', 'market_size': 'market size'},
    template='ggplot2'
)

fig.show()

In [ ]:
sales_by_market = (
    df.groupby(['promotion', 'market_size'], as_index=False)['sales_in_thousands']
    .sum()
)

fig = px.bar(
    sales_by_market,
    x='promotion',
    y='sales_in_thousands',
    color='market_size',
    barmode='stack',
    title='breakdowns of market sizes across different promotions',
    labels={'promotion': 'Promotion', 'sales_in_thousands': 'Sales (in Thousands)', 'market_size': 'Market Size'},
    template='ggplot2'
)

fig.show()

#### - Store Age

In [ ]:
df['age_of_store'].describe()

In [ ]:
age_counts = df.groupby('age_of_store', as_index=False)['market_id'].count().rename(columns={'market_id': 'count'})

fig = px.bar(
    age_counts,
    x='age_of_store',
    y='count',
    title='overall distributions of age of store',
    labels={'age_of_store': 'age', 'count': 'count'},
    color_discrete_sequence=['skyblue'],
    template='ggplot2'
)

fig.show()


In [ ]:
age_promo_counts = (
    df.groupby(['age_of_store', 'promotion'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('age_of_store', ascending=False)
)

age_promo_counts['promotion'] = age_promo_counts['promotion'].astype(str)

fig = px.bar(
    age_promo_counts,
    x='count',
    y='age_of_store',
    color='promotion',
    orientation='h',
    barmode='group',
    title='overall distributions of age of store',
    labels={'age_of_store': 'age', 'count': 'count', 'promotion': 'promotion'},
    template='ggplot2',
    category_orders={'promotion': ['1', '2', '3']},
    height=800
)
fig.update_yaxes(autorange='reversed')
fig.show()

In [ ]:
df.groupby('promotion').describe()['age_of_store']

#### - Week Number

In [ ]:
df.groupby('week').count()['market_id']

In [ ]:
df.groupby(['promotion', 'week']).count()['market_id']

In [ ]:
from plotly.subplots import make_subplots

import plotly.graph_objects as go

week_promo_counts = (
    df.groupby(['promotion', 'week'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['Promotion #1', 'Promotion #2', 'Promotion #3']
)

for i, promo in enumerate([1, 2, 3], start=1):
    subset = week_promo_counts[week_promo_counts['promotion'] == promo]
    fig.add_trace(
        go.Pie(
            labels=subset['week'],
            values=subset['count'],
            name=f'Promotion #{promo}',
            textinfo='percent',
            hovertemplate='Week %{label}<br>Count %{value}<br>%{percent}<extra></extra>'
        ),
        row=1,
        col=i
    )

fig.update_layout(
    title='distribution across different weeks',
    template='ggplot2',
    showlegend=True
)

fig.show()

# 3. Statistical Significance

In [ ]:
import numpy as np
from scipy import stats

#### - t-test

In [ ]:
df.info()

In [ ]:
means = df.groupby('promotion')['sales_in_thousands'].mean()
means

In [ ]:
stds = df.groupby('promotion')['sales_in_thousands'].std()
stds

In [ ]:
ns = df.groupby('promotion')['sales_in_thousands'].count()
ns

#### - Promotion 1 vs. 2

In [ ]:
t_1_vs_2 = (
    means.iloc[0] - means.iloc[1]
)/ np.sqrt(
    (stds.iloc[0]**2/ns.iloc[0]) + (stds.iloc[1]**2/ns.iloc[1])
)

df_1_vs_1 = ns.iloc[0] + ns.iloc[1] - 2

p_1_vs_2 = (1 - stats.t.cdf(t_1_vs_2, df=df_1_vs_1))*2

In [ ]:
t_1_vs_2

In [ ]:
p_1_vs_2

#### - using scipy

In [ ]:
t, p = stats.ttest_ind(
    df.loc[df['promotion'] == 1, 'sales_in_thousands'].values, 
    df.loc[df['promotion'] == 2, 'sales_in_thousands'].values, 
    equal_var=False
)

In [ ]:
t

In [ ]:
p

#### - Promotion 1 vs. 3

In [ ]:
t_1_vs_3 = (
    means.iloc[0] - means.iloc[2]
)/ np.sqrt(
    (stds.iloc[0]**2/ns.iloc[0]) + (stds.iloc[2]**2/ns.iloc[2])
)

df_1_vs_3 = ns.iloc[0] + ns.iloc[1] - 2

p_1_vs_3 = (1 - stats.t.cdf(t_1_vs_3, df=df_1_vs_3))*2

In [ ]:
t_1_vs_3

In [ ]:
p_1_vs_3

#### - using scipy

In [ ]:
t, p = stats.ttest_ind(
    df.loc[df['promotion'] == 1, 'sales_in_thousands'].values, 
    df.loc[df['promotion'] == 3, 'sales_in_thousands'].values, 
    equal_var=False
)

In [ ]:
t

In [ ]:
p